# 02 · Curate: frames out of the recording

**Agenda: 25–40 min.** Detectors train on images. The RGB cameras inside each MCAP are H.264 streams, so `build/extract_frames.py` decodes them through ffmpeg at 1 fps, resizes to 640 px, and writes a FiftyOne image dataset where every frame remembers its **episode, camera, timestamp**, and the robot state at that instant (gripper open/closed, joint speed).

You are loading the result: 5,172 frames from the 75 episodes that carry video.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F
import fiftyone.utils.huggingface as fouh

def load(name):
    # Local copy if download_data.py already ran, otherwise pull it from Hugging Face now
    if fo.dataset_exists(name):
        return fo.load_dataset(name)
    print(f"{name} not found locally; downloading from Hugging Face (one time)…")
    return fouh.load_from_hub(f"dgural/{name}", name=name, persistent=True)

frames = load("droid-frames-workshop")
print(len(frames), "frames from", len(frames.distinct("episode_id")), "episodes")
print(frames.count_values("camera"))
frames

In [ ]:
session = fo.launch_app(frames)

### The extraction, in one cell (reference — do not run on the full set now)

```python
# build/extract_frames.py, the heart of it
proc = subprocess.Popen(["ffmpeg", "-f", "h264", "-i", "pipe:0",
                         "-vf", f"fps={fps},scale={width}:-2", "-q:v", "3", out_pattern], stdin=subprocess.PIPE)
for _, channel, message, proto in reader.iter_decoded_messages(topics=["/camera/wrist/image_rgb"]):
    proc.stdin.write(proto.data)          # raw H.264 NAL units → ffmpeg
```

Each output frame becomes an `fo.Sample` with `episode_id`, `camera`, `timestamp_ns`, `gripper_open`, `joint_speed_norm`. That link back to the recording is what makes notebook 06 possible.

## Filter with robot state, not just pixels

Frames where the gripper is closed are the ones where something is being held — the interesting ones for a gripper/object detector.

In [ ]:
closed = frames.match(F("gripper_open") == False)  # noqa: E712
print(len(closed), "frames with the gripper closed")

moving = frames.match(F("joint_speed_norm") > 0.8)
print(len(moving), "frames while the arm is moving fast (motion blur candidates)")

session.view = closed.match(F("camera") == "wrist")

## Uniqueness and near-duplicates

1 fps on a slow-moving arm produces many near-identical frames. `compute_uniqueness` (precomputed) scores how unusual each frame is relative to the rest; frames the similarity index flagged as near-duplicates carry the tag `near-duplicate`.

In [ ]:
print(frames.bounds("uniqueness"))
print(frames.count_sample_tags())

# Least unique frames: the bench with nothing happening
session.view = frames.sort_by("uniqueness")

In [ ]:
# Most unique: odd viewpoints, hands in frame, dropped objects
session.view = frames.sort_by("uniqueness", reverse=True)

## Build the curation view

Keep external-camera frames, drop near-duplicates, prefer the ones where the gripper is closed or the arm is moving. Save it — a saved view is a reusable query, not a copy of the data.

In [ ]:
curated = (frames
           .match_tags("near-duplicate", bool=False)
           .match(F("camera") != "wrist")
           .match((F("gripper_open") == False) | (F("joint_speed_norm") > 0.3)))  # noqa: E712
print(len(curated), "frames in the curated view")

frames.save_view("curated_train", curated, overwrite=True)
session.view = curated

**Optional (≈1 min on CPU):** recompute uniqueness yourself on a 300-frame slice to see how it works.

In [ ]:
# OPTIONAL — ~1 min on CPU
# import fiftyone.brain as fob
# slice_ = frames.take(300, seed=51)
# fob.compute_uniqueness(slice_, uniqueness_field="uniqueness_mine")
# session.view = slice_.sort_by("uniqueness_mine", reverse=True)

**Next:** the curated view is what we embed, label and train on. Notebook 03 looks at it through embeddings.